<a href="https://colab.research.google.com/github/laulivette/poc_indutech/blob/main/poc_indutech.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Étape 1 — Configurez Redpanda

### 1.1 Installation de Redpanda

In [6]:
%%bash
# Installation de Redpanda (paquet officiel .deb)
curl -1sLf 'https://dl.redpanda.com/nzc4ZYQK3WRGd9sy/redpanda/cfg/setup/bash.deb.sh' | sudo -E bash > /dev/null 2>&1
sudo apt-get install redpanda -y -qq > /dev/null 2>&1
echo "Redpanda installe avec succes"
rpk version

Redpanda installe avec succes
rpk version: v26.2.1
Git ref:     8cd781be99c737c7d147d9dc239a0bbdd591a4d8
Build date:  2026 Jul 24 10 13 57 Fri
OS/Arch:     linux/amd64
Go version:  go1.26.5

Redpanda Cluster
  Unreachable, to debug, use the '-v' flag. To get the broker versions, pass the
  hosts via flags, profile, or environment variables:
    rpk version -X admin.hosts=<host address>

  To get only the rpk version, use 'rpk --version'.


### 1.2 Lancement de Redpanda

In [7]:
import subprocess, time

# Lancement Redpanda en arriere-plan (mode dev, un seul noeud)
proc = subprocess.Popen(
    ["sudo", "rpk", "redpanda", "start",
     "--mode", "dev-container",
     "--smp", "1",
     "--memory", "1G",
     "--overprovisioned",
     "--default-log-level=info"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

time.sleep(8)  # laisse le temps au cluster de demarrer
print("Redpanda demarre (PID:", proc.pid, ")")

Redpanda demarre (PID: 8245 )


In [8]:
# Verification que le cluster est operationnel
!rpk cluster info

CLUSTER
redpanda.1a4a16d8-a479-434a-b4ce-e5ba1516ea3e

BROKERS
ID    HOST       PORT
0*    127.0.0.1  9092



### 1.3 Création du topic `client_tickets`

In [9]:
!rpk topic create client_tickets --partitions 3 --replicas 1
!rpk topic list

TOPIC           STATUS
client_tickets  OK
NAME            PARTITIONS  REPLICAS
client_tickets  3           1


### 1.4 Script producteur — génération de tickets aléatoires

On installe `confluent-kafka` (compatible avec l'API Kafka exposée par Redpanda) puis on écrit un générateur de tickets.

In [10]:
!pip install confluent-kafka faker -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 97.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 80.6 MB/s eta 0:00:00


In [11]:
import json
import random
import uuid
from datetime import datetime, timezone
from confluent_kafka import Producer
from faker import Faker

fake = Faker("fr_FR")

TOPIC = "client_tickets"
BOOTSTRAP_SERVERS = "localhost:9092"

TYPES_DEMANDE = [
    "Probleme technique",
    "Facturation",
    "Demande d'information",
    "Reclamation",
    "Resiliation",
    "Support produit",
]

PRIORITES = ["Basse", "Moyenne", "Haute", "Critique"]

DEMANDES_TEMPLATES = {
    "Probleme technique": "Le service ne repond plus depuis ce matin.",
    "Facturation": "Le montant preleve ne correspond pas au contrat.",
    "Demande d'information": "Je souhaite en savoir plus sur l'offre premium.",
    "Reclamation": "Le delai de livraison n'a pas ete respecte.",
    "Resiliation": "Je souhaite resilier mon abonnement.",
    "Support produit": "Je n'arrive pas a configurer le produit.",
}

def generer_ticket():
    type_demande = random.choice(TYPES_DEMANDE)
    return {
        "ticket_id": str(uuid.uuid4()),
        "client_id": fake.uuid4(),
        "created_at": datetime.now(timezone.utc).isoformat(),
        "demande": DEMANDES_TEMPLATES[type_demande],
        "type_demande": type_demande,
        "priorite": random.choice(PRIORITES),
    }

def delivery_report(err, msg):
    if err is not None:
        print(f"Echec de livraison: {err}")

producer = Producer({"bootstrap.servers": BOOTSTRAP_SERVERS})

print("Envoi des tickets vers le topic", TOPIC)


Envoi des tickets vers le topic client_tickets


In [12]:
import time

NB_TICKETS = 200          # nombre de tickets a generer
INTERVALLE_SEC = 0.05      # delai entre deux envois (simulateur temps reel)

for i in range(NB_TICKETS):
    ticket = generer_ticket()
    producer.produce(
        TOPIC,
        key=ticket["ticket_id"],
        value=json.dumps(ticket).encode("utf-8"),
        callback=delivery_report,
    )
    producer.poll(0)
    time.sleep(INTERVALLE_SEC)

producer.flush()
print(f"{NB_TICKETS} tickets envoyes dans le topic '{TOPIC}'")


200 tickets envoyes dans le topic 'client_tickets'


In [13]:
# Verification rapide : consommation de quelques messages depuis le topic
!rpk topic consume client_tickets --num 3 --format json

{
  "topic": "client_tickets",
  "key": "78f12a1c-3a3e-47a0-9db3-41732db983f4",
  "value": "{\"ticket_id\": \"78f12a1c-3a3e-47a0-9db3-41732db983f4\", \"client_id\": \"54deb28b-ba01-43ea-9487-7780d22f2d6b\", \"created_at\": \"2026-08-05T07:42:33.428171+00:00\", \"demande\": \"Le service ne repond plus depuis ce matin.\", \"type_demande\": \"Probleme technique\", \"priorite\": \"Basse\"}",
  "timestamp": 1785915753428,
  "partition": 2,
  "offset": 0
}
{
  "topic": "client_tickets",
  "key": "bca4db22-c80d-48a1-b11b-b9853f4c6e27",
  "value": "{\"ticket_id\": \"bca4db22-c80d-48a1-b11b-b9853f4c6e27\", \"client_id\": \"99cb8509-31b7-46e6-833d-25ce71f78d9f\", \"created_at\": \"2026-08-05T07:42:33.529251+00:00\", \"demande\": \"Le montant preleve ne correspond pas au contrat.\", \"type_demande\": \"Facturation\", \"priorite\": \"Haute\"}",
  "timestamp": 1785915753529,
  "partition": 2,
  "offset": 1
}
{
  "topic": "client_tickets",
  "key": "8f748a62-4287-4b3c-8c7e-edece5e13542",
  "value": 